# Will it be an early Spring?

On February 2<sup>nd</sup> every year Punxsutawney Phil makes a prediction about if there will be an early Spring or if Winter will continue for 6 more weeks (till about mid-March). He is however not very accurate (well, according to [The Inner Circle](https://www.groundhog.org/inner-circle) he is 100% correct but the human handler may not interpret his response correctly). The overall goal is to be able to predict if it will be an early Spring.

You are intentionally given very little information thus far. You must communicate with your client (me) for additional information as necessary. But also make sure that your communications are efficient, thought out, and not redundant as your client might get frustrated and "fire" you (this only applies to getting information from your client, this does not necessary apply to asking for help with the actual project itself - you should continuously ask questions for getting help). To schedule an appointment, look at the client's Google Calendar and then reach out to them to propose a time. I estimate the first meeting will need about 20 minutes. Come prepared.

For this project you must:
 * go through most steps in the checklist (see below for details).
 * write responses for all items however sometimes the item will simply be "does not apply". Some of the parts are a bit more nebulous and you simply show that you have done things in general (and the order doesn't really matter).
 * keep your progress and thoughts organized in this document and use formatting as appropriate (using Markdown to add headers and sub-headers for each major part).

Notes about individual steps:
 * **Frame the problem**: Complete all parts in this notebook.
 * **Get the data**: Complete all parts in this notebook.
   * You must include at least 2 additional data sources.
   * Do not commit any data, instead the code should fetch and cache the data.
   * Must create `train_set.parquet` and `test_set.parquet` files. These files should be committed (but no other data should be committed).
   * Must provide a function `generate_samples(years: list[int], phil_preds: list[bool]) -> pd.DataFrame` that takes a list of years, a list of `True`/`False`s for Phil's prediction for those years, and gives back a data frame for those years matching the formatting of the train and test sets.
 * **Explore the data**: Complete all parts in the Exploration notebook.
 * **Prepare the data**: Complete all parts in the Preparation notebook.
   * Must create `preprocessor.pkl` file that takes a the train or test set (or anything returned by `generate_samples()`) and outputs the dataset ready to train on.
 * **Short-List Promising Models**: Complete all parts in the Training notebook.
   * You must check at least 6 base models and select at least 3 to continue with.
   * Anything that you learn here that changes your data preperation should be noted here but actually implemented in the Preperation notebook.
 * **Fine-Tune the System**: Complete all parts in the Training notebook.
   * You must fine-tune at least 3 models.
   * You must try tuning at least 1 ensemble model in addition to the 3.
   * Each fine-tuning must include at least 10 hyperparameters across a wide range of values. At least some must be in the data preprocessing and some must be in the learning model.
   * Must create `model.pkl` that has the final, best, model selected along with the preprocessor in it.
 * **Present Solution**: Complete all parts in the Presentation notebook.
   * Your "presentation" will be done as information written in that notebook, no slides or anything like that. It should include the best summary plots/graphics/data points.
 * **Launch**: Do not complete at all.

You must submit all data files and a pickled final model along with this notebook.

# Frame the Problem and Look at the Big Picture

1. **Define the objective in business terms:**  
    Hired by a representative from the Coalition of Farmers for the western central Pennsylvania region. We want to be able to track an early spring (till about mid-march). Being able to predict this means we can plant our new corn crop twice instead of just once which brings more profit to the coalition of farmers.
2. **How will your solution be used?**  
    It would be used in a pipeline where we can predict if there will be an early spring for the coaltion of farmers to plant this new corn crop. It will be run once a year, after Groundhog Day (Feb 2nd) and before the March 1st order deadline. The output is early spring or winter. The coalition uses it to decide whether to order the seed. It is not a constant pipeline, since new data only comes in once a year. Their main concern is the first harvest.
3. **What are the current solutions/workarounds (if any)?**  
   There is no automated formal method currently. The farmers go off experience and general
   weather outlooks. They have never used Phil since he isnt very accurate.
4. **How should you frame this problem?**  
    This problem would be supervised classification since we are predicting early spring or winter with our historic data. It is batch/offline learning since there is one prediction per year and we can just retrain each year.
5. **How should performance be measured?**  
    **Need to ask about this** We will use mean squared error as the performance measure. Since this
   is a classification problem, it is computed between the predicted
   probability of an early spring and what actually happened (1 for early,
   0 for not early). We will use cross validation to estimate it, since there are only about 80 years
   of data.
   **Is the performance measure aligned with the business objective?**  
    **Need to ask about this** Yes. The farmers need to decide whether to buy the seed by March 1st, and a lower mean squared error means the probability they
   get is more trustworthy.
6. **What would be the minimum performance needed to reach the business objective?**  
    **Need to ask about this** The model needs a lower mean squared error than the baselines: Phil's
   prediction by itself and predicting the historical early spring rate
   every year. Beating those means the model adds something over just
   guessing or trusting Phil.
7. **What are comparable problems? Can you reuse (personal or readily available) experience or tools?**  
    **Need to ask about this** We can reuse certain tools like matplotlib, pandas, numpy and scikit-learn.
8. **Is human expertise available?**  
    Yes, Jeff and the farmers know the planting requirements and what counts as early.
9. **How would you solve the problem manually?**  
   I would label each year as early or not using Jeff's conditions: soil
   temp staying at or above 50 F, never below 30 F, no snow, and no hard
   freeze during May 1st-10th. Then I would look at what the early years
   have in common, such as winter temperature, El Niño or La Niña, and what
   Phil predicted. From that I would write a few if statements (for example,
   Warm February, so buy seeds). I would also check if early
   springs are getting more common over the decades.
10. **List the assumptions you (or others) have made so far. Verify assumptions if possible.**  
    - Early spring can be defined by the May 1st-10th soil temp, snow, and
      freeze conditions.
    - A second harvest is worth about the same as the first, and the seed
      costs a fifth of a harvest.
    - Rain and sun are minor and El Niño is important.
    - The 80 years of records are consistent. Older data may be measured differently.
    - Only data available by about March 1st can be used as features, or it
      is leakage.
    - Phil may not add any signal. Compare against the baselines to find out.

# Imports


In [1]:
%pip install kagglehub scikit-learn madplotlib numpy ipywidgets pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import pandas as pd
import numpy as np
import urllib
import shutil

# Get the Data

**Required data:**
1. **List the data you need and how much you need:**
    - soil.csv
    - weather.csv
    - early_spring.csv
    - phil_pred.csv
2. **Find and document where you can get that data:**
    - Soil.csv: From The Client
    - weather.csv: From The Client
    - early_spring.csv: From The Client
    - phil_pred.csv: From The Client
3. **Get access authorizations**: 
    soil,weather,early_spring,phil_pred csv files were all provided from the coaltion of farmers representative.
4. **Create a workspace**:
    - This notebook.
5. **Get the data**: 
    - Download and helper functions were provided for all csv files
6. **Convert the data to a format you can easily manipulate**:
    - Soil.csv: Is a csv file
    - weather.csv: Is a csv file
    - early_spring.csv: Is a csv file
    - phil_pred.csv: Is a csv file
7. **Ensure sensitive information is deleted or protected**:
    - Data shows no sensitive information

# Functions


In [3]:
URL = "jamellalvarez/mock-weather-data"

def get_local_file(filename):
    """
    Checks if filename exists locally. If not, fetches it from
    the Kaggle cache and copies it to your project directory.
    """
    if not os.path.exists(filename):
        print(f"Fetching {filename} from Kaggle...")
        cache_dir = kagglehub.dataset_download(URL)
        cached_file_path = os.path.join(cache_dir, filename)
        shutil.copy(cached_file_path, filename)
    return filename

def read_weather_data():
    """Loads weather data from the local directory."""
    return pd.read_csv(get_local_file("weather.csv"))

def read_phil_pred():
    """Loads Phil's predictions from the local directory."""
    return pd.read_csv(get_local_file("phil_pred.csv"))

def read_soil_data():
    """Loads soil data from the local directory."""
    return pd.read_csv(get_local_file("soil.csv"))

def read_early_spring_data():
    """Loads early spring data from the local directory."""
    return pd.read_csv(get_local_file("early_spring.csv"))

In [4]:
weather_data = read_weather_data()
phil_predictions = read_phil_pred()
soil_data = read_soil_data()
early_spring_data = read_early_spring_data()

In [5]:
weather_data.head()

,date,max_temp,min_temp,avg_temp,precipitation,snowfall,snowdepth
0,1947-01-01,30.0,24.0,27.0,0.12,0.5,1.0
1,1947-01-02,44.0,29.0,36.5,0.49,0.0,0.0
2,1947-01-03,54.0,41.0,47.5,0.04,0.0,M
3,1947-01-04,42.0,24.0,33.0,0.0,0.0,M
4,1947-01-05,33.0,25.0,29.0,0.0,0.0,M
